In [1]:
import json
import pandas as pd
import glob
import os
import re

# Get the current working directory
current_directory = os.getcwd()

# Set the new working directory
new_directory = '/Users/lzz/Documents/GitHub/IVF/data_cleaning/code'
os.chdir(new_directory)

In [3]:
# Updated function to process the PatientIDx
def process_embryo_trans(embryo_trans):
    # Remove the prefix 'treatments_'
    embryo_trans = embryo_trans.replace('transfers_', '')
    
    # Replace the last underscore with a dot
    embryo_trans = re.sub(r'_(?=[^_]*$)', '.', embryo_trans)
    
    return embryo_trans

# Get a list of all JSON files in the current directory
json_files = glob.glob('../data/transfers/*.json')

# Initialize an empty list to store the DataFrames
dfs = []

# Loop through each JSON file
for file in json_files:
    # Extract the file name without extension to use as the PatientID
    patient_id = os.path.basename(file)
    patient_id = file.split('.')[0]
    
    # Read the data from the JSON file
    with open(file, 'r', encoding='utf-8') as f:
        ids_data = json.load(f)
        
        # Extract the TreatmentList and create a DataFrame
        df = pd.DataFrame(ids_data['TransferList'], columns=['Transfers'])
        
        # Add the PatientIDx as a new column
        df['PatientIDx'] = patient_id
        
        # Process the PatientIDx
        df['PatientIDx'] = df['PatientIDx'].apply(process_embryo_trans)

        # Append the DataFrame to the list
        dfs.append(df)

# Concatenate all DataFrames into a single DataFrame
df_all = pd.concat(dfs, ignore_index=True)

# Display the final DataFrame
# print(df_all)

# Save the DataFrame to a CSV file
df_all.to_csv('../output/embryo_tranfers_clean.csv', index=False, encoding='utf-8-sig')